# V5: train ONE unified seven-class YOLO11s on original Day1 + V4

This is **one seven-class model**, not a V1 + V4 inference hybrid.

- Training: **2,093 photos** = original V1-era Day1 archive 1,572 (excluding condiment packets) + V4 cafeteria train 521.
- Validation: **97 held-out V4 images**, containing labeled **food, plate, utensil, wrapper** only.
- Historical test: 71 reserved V4 images, **never evaluated or tuned here**.
- New seven-class names: `food, plate, utensil, hinged_container, drink_carton, wrapper, bottle`.
- 161 V4 legacy images duplicated in V1 Day1 are excluded. Source provenance and SHA256 hashes are in the dataset manifest.

**Warning:** The source datasets have incomplete cross-class annotations. The 97-image V4 validation only has labels for four classes. Therefore any validation `mAP50` / `mAP50-95` is a **four-represented-class** score, not seven-class accuracy. The original Day1 archive may not be the identical dataset used to train the frozen four-specialist V1 system.

Use **Runtime → Change runtime type → T4 GPU**. This notebook does not alter V1 or V4 weights.


## 1. Install packages and verify GPU


In [ ]:
import sys,subprocess,hashlib,json,zipfile,torch
from pathlib import Path
from google.colab import files
assert torch.cuda.is_available(),"Choose a T4 GPU in Colab Runtime settings"
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.175"],check=True)
from ultralytics import YOLO
print("GPU:",torch.cuda.get_device_name(0),flush=True)
def file_sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1048576),b""):h.update(b)
    return h.hexdigest()


## 2. Upload the prepared V5 dataset ZIP (217 MB)


In [ ]:
zipname="YOLO_V5_original_plus_V4_7class.zip"
archive=Path("/content")/zipname
EXPECTED="12a9c3d7db44adf887a915e09c876941a407730a429913d04f82498dce285e4c"
if not archive.exists():
    print("Upload",zipname,flush=True)
    upload=files.upload()
    assert zipname in upload,"Upload the exact V5 dataset ZIP"
    archive.write_bytes(upload[zipname])
assert file_sha(archive)==EXPECTED,"V5 archive checksum mismatch"
root=Path("/content/v5_original_plus_v4")
if not (root/"manifest.json").exists():
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None,"Corrupt archive"
        assert all(Path(i.filename).parts[:1]==("v5_original_plus_v4",) for i in z.infolist())
        z.extractall("/content")
manifest=json.loads((root/"manifest.json").read_text())
from collections import Counter
counts=Counter(r["split"] for r in manifest)
assert counts=={"train":2093,"val":97,"test":71},counts
report=json.loads((root/"merge_report.json").read_text())
print("Splits:",dict(counts),flush=True)
print("Training boxes:",report["class_boxes_by_split"]["train"],flush=True)
print("Validation boxes:",report["class_boxes_by_split"]["val"],flush=True)
print("NOTE: only 4/7 classes have labels in validation.",flush=True)


## 3. Train a NEW seven-class YOLO11s checkpoint

Starts from generic pretrained `yolo11s.pt`. Moderate augmentation, 40 maximum epochs, early stopping 12, images 800px. A larger, mixed-source dataset generally needs more training than V4's 20-epoch run. Compare validation metrics before deciding what to change.


In [ ]:
project=Path("/content/V5_unified_results")
assert not project.exists(),"Results folder exists. Avoid overwriting; restart runtime or choose a new folder."
model=YOLO("yolo11s.pt")
model.train(
    data=str(root/"data.yaml"),
    epochs=40, patience=12, imgsz=800, batch=4, device=0,
    workers=2, project=str(project), name="train", exist_ok=False,
    optimizer="AdamW", lr0=0.0005, lrf=0.01, weight_decay=0.0005,
    cos_lr=True, warmup_epochs=3, seed=42, deterministic=True,
    degrees=10, translate=0.08, scale=0.25,
    fliplr=0.5, flipud=0, mosaic=0.5, mixup=0, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.35, hsv_v=0.25,
    cache=False, plots=True
)
best=project/"train/weights/best.pt"
assert best.is_file(),"No trained best.pt checkpoint created"
print("Checkpoint:",best,flush=True)


## 4. Measure held-out validation mAP50 and mAP50–95

These scores only apply to classes **present in the 97-image validation labels**. Do not describe them as seven-class mAP, and do not directly compare them with the historical V1 seven-class test results.


In [ ]:
trained=YOLO(str(best))
metric=trained.val(
    data=str(root/"data.yaml"), split="val", imgsz=800,
    batch=4, device=0, conf=0.001, iou=0.7, plots=False
)
classes=[str(trained.names[int(i)]) for i in metric.box.ap_class_index]
assert set(classes)=={"food","plate","utensil","wrapper"},classes
per_class={name:{
    "AP50":round(float(metric.box.ap50[j]),6),
    "AP50_95":round(float(metric.box.ap[j]),6)}
    for j,name in enumerate(classes)}
summary={
  "training_images":2093,"validation_images":97,
  "model":"YOLO11s", "num_output_classes":7,
  "validation_classes_present":classes,
  "mAP50_four_present_classes":round(float(metric.box.map50),6),
  "mAP50_95_four_present_classes":round(float(metric.box.map),6),
  "per_class":per_class,
  "seven_class_mAP_measured":False,
  "test_evaluated":False,
  "checkpoint_sha256":file_sha(best),
  "annotation_completeness": "not independently confirmed"
}
out=project/"validation_metrics.json"
out.write_text(json.dumps(summary,indent=2)+"\n")
print(json.dumps(summary,indent=2),flush=True)


## 5. Export weights and metrics (no test run)


In [ ]:
output=Path("/content/YOLO_V5_trained_results.zip")
with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for rel in ["train/weights/best.pt","train/results.csv",
                "train/args.yaml","validation_metrics.json"]:
        file=project/rel
        if file.is_file():z.write(file,arcname=rel)
print("Saved",output,round(output.stat().st_size/1048576,1),"MiB",flush=True)
files.download(str(output))
